# 04 - The surrogate as a forward operator

§11.2 step 7's gates, measured on the test split, and the three figures §11.3 calls
non-negotiable for the forward model:

1. predicted vs true scattered wavefield, for a defect the network never saw;
2. predicted vs true at all 32 receivers -- the only part of the field the inversion reads;
3. error against `|k|`, which says *where* in the spectrum the remaining error lives.

The test split is the honest one. `generate` gives train and val the reduced source pool
`SRC_TRAIN`, and gives test **all** sources, so `SRC_HELDOUT = (3, 6)` appears in test and
nowhere else. Every number below is therefore reported twice: over trained illuminations, and
over the two the network has never been shown. A surrogate that has memorised eight source
patterns rather than learned an operator separates cleanly on that split, and no amount of
held-out *geometry* would reveal it.

Read-only: no training, no solving. Runs in a couple of minutes.

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later notebooks read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

In [ ]:
from src import features as feat
from src import losses as L
from src import training
from src.data.dataset import WaveDataset, batch_to_model, make_loader, to_device
from src.models.fno2d import band_in_modes
from src.solver import harmonic as H

assert paths["test"].exists(), "run notebook 02 first"
assert CKPT.exists(), f"no checkpoint at {CKPT} -- run notebook 03 first"

model, meta = training.load(CKPT, device=DEV)
model.eval()
print(f"loaded {CKPT}")
print(f"  arch  {meta['arch']}")
print(f"  epoch {meta['epoch']}   val {meta['val']}   alpha {meta['alpha']}")
print(f"  {model.effective_params():,} effective parameters")

## Gates on the test split

`evaluate` puts the dataset in eval mode, which turns off the frequency subsetting: every
sample contributes all `M_FREQ = 20` lines, so a batch of 4 samples is 80 rows through the
network. The per-frequency breakdown comes from the same pass.

In [ ]:
ts = WaveDataset(str(paths["test"]), train=False)
tl = make_loader(ts, batch_size=4, shuffle=False, num_workers=0)
print(f"test split: {len(ts)} samples, {ts.n_freq} frequencies each, "
      f"{len(ts) * ts.n_freq} rows")

t0 = time.perf_counter()
ev = training.evaluate(model, tl, DEV, per_freq=True)
print(f"\nevaluated in {time.perf_counter()-t0:.1f} s\n")
print(ev)

### Per-sample errors, split by illumination

`evaluate` returns means. The means are what the gates are stated on, but they hide the tail,
and the tail is what the inversion trips over: one sample at 30% error is a failed inversion
even if the mean is 3%. The loop below recomputes the same two metrics per sample so the
distribution, the worst cases and the held-out-source comparison are all available.

In [ ]:
recv = training.receivers_tensor(DEV)


@torch.no_grad()
def per_sample(loader):
    rows = []
    for batch in loader:
        b = to_device(batch, DEV)
        x, y = batch_to_model(b)
        p = model(x)
        nf = b["freqs"].shape[1]
        # rows are (sample, frequency), frequency-major
        d = (p - y).reshape(-1, nf, *y.shape[1:])
        t = y.reshape(-1, nf, *y.shape[1:])
        dims = (1, 2, 3, 4)
        f_err = (d.pow(2).sum(dims).sqrt() / t.pow(2).sum(dims).sqrt()).cpu()
        ry, rx = recv[:, 0], recv[:, 1]
        dr, tr = d[..., ry, rx], t[..., ry, rx]
        r_err = (dr.pow(2).sum(dims).sqrt() / tr.pow(2).sum(dims).sqrt()).cpu()
        for k in range(f_err.shape[0]):
            rows.append(dict(index=int(b["index"][k]), src=int(b["src_idx"][k]),
                             nu_idx=int(b["nu_idx"][k]),
                             theta=_np(b["theta"][k]).tolist(),
                             field=float(f_err[k]), ring=float(r_err[k])))
    return rows


ps = per_sample(tl)
fe = np.array([r["field"] for r in ps])
re_ = np.array([r["ring"] for r in ps])
src = np.array([r["src"] for r in ps])
held = np.isin(src, cfg.SRC_HELDOUT)

table([("all", f"{len(fe)}", f"{fe.mean():.4f}", f"{np.median(fe):.4f}",
        f"{np.percentile(fe, 95):.4f}", f"{fe.max():.4f}", f"{re_.mean():.4f}"),
       ("trained sources", f"{(~held).sum()}", f"{fe[~held].mean():.4f}",
        f"{np.median(fe[~held]):.4f}", f"{np.percentile(fe[~held], 95):.4f}",
        f"{fe[~held].max():.4f}", f"{re_[~held].mean():.4f}"),
       (f"held out {cfg.SRC_HELDOUT}", f"{held.sum()}", f"{fe[held].mean():.4f}",
        f"{np.median(fe[held]):.4f}", f"{np.percentile(fe[held], 95):.4f}",
        f"{fe[held].max():.4f}", f"{re_[held].mean():.4f}")],
      ["subset", "n", "mean", "median", "p95", "max", "ring mean"])

pen = fe[held].mean() / max(fe[~held].mean(), 1e-30) - 1.0
print(f"\nheld-out illumination costs {pen:+.1%} on the mean field error")
print(f"gate rel-L2 < {cfg.GATE_REL_L2:.0%}: "
      f"{'PASS' if fe[held].mean() < cfg.GATE_REL_L2 else 'FAIL'} even on held-out sources"
      if held.any() else "")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11.2, 3.0))

bins = np.linspace(0, max(fe.max(), cfg.GATE_REL_L2 * 1.5), 40)
ax[0].hist(fe[~held], bins=bins, alpha=0.7, label="trained sources", density=True)
if held.any():
    ax[0].hist(fe[held], bins=bins, alpha=0.7, label=f"held out {cfg.SRC_HELDOUT}",
               density=True)
ax[0].axvline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0, label=f"gate {cfg.GATE_REL_L2:.0%}")
ax[0].set(xlabel="field rel-L2", ylabel="density", title="per-sample field error")
ax[0].legend(fontsize=7.5)

R = np.array([r["theta"][2] for r in ps])
lam = np.array([cfg.cs_over_cp(cfg.NU_LIST[r["nu_idx"]]) / cfg.FC for r in ps])
ax[1].plot(R / lam, fe, ".", ms=3, alpha=0.5)
ax[1].axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0)
ax[1].axvline(cfg.R_MIN_LS, ls=":", c="0.4", lw=1.0)
ax[1].set(xlabel="R / lambda_s", ylabel="field rel-L2",
          title="error vs defect size\n(small voids scatter least, so relative\n"
                "error is hardest there)")

sx = np.array([cfg.SOURCE_XY[r["src"]][0] for r in ps])
sy = np.array([cfg.SOURCE_XY[r["src"]][1] for r in ps])
dist = np.hypot(np.array([r["theta"][0] for r in ps]) - sx,
                np.array([r["theta"][1] for r in ps]) - sy) / lam
ax[2].plot(dist, fe, ".", ms=3, alpha=0.5)
ax[2].axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0)
ax[2].set(xlabel="source-defect distance / lambda_s", ylabel="field rel-L2",
          title="error vs standoff")
fig.tight_layout()
savefig(fig, "04_error_distribution.png")
plt.show()

## Figure 1 -- the predicted wavefield

One test sample, at the bottom, middle and top of the band. `|u_s|` for truth and prediction
share a colour scale per row; the third column is `|u_s_pred - u_s_true|` on the *same* scale,
so a visible error panel is a real error and not a rescaled one. The void outline is the
`chi = 0.5` contour of the same soft indicator the network was given as input, and the source
is marked.

The sample shown is the **worst held-out-source case** by field error, because a montage of a
median case is a picture of the network working and a montage of the worst case is the only
one that can show *how* it fails. If the error concentrates on the void boundary, that is the
interface width; if it trails behind the scattered front, that is dispersion; if it sits at the
domain edge, that is the absorber leaking into the labels.

In [ ]:
pool = np.where(held)[0] if held.any() else np.arange(len(ps))
pick = int(pool[np.argmax(fe[pool])])
rec = ps[pick]
print(f"sample {rec['index']}  src {rec['src']} "
      f"{'(HELD OUT)' if rec['src'] in cfg.SRC_HELDOUT else ''}  "
      f"nu {cfg.NU_LIST[rec['nu_idx']]}  theta {np.round(rec['theta'], 4)}  "
      f"field rel-L2 {rec['field']:.4f}")

one = WaveDataset(str(paths["test"]), train=False)
b = to_device({k: (v.unsqueeze(0) if torch.is_tensor(v) else v)
               for k, v in one[rec["index"]].items()}, DEV)
x1, y1 = batch_to_model(b)
with torch.no_grad():
    p1 = model(x1)
zt = _np(feat.channels_to_complex(y1))          # [M, 2, ny, nx] complex
zp = _np(feat.channels_to_complex(p1))
chi1 = _np(b["chi"][0])
sy_, sx_ = cfg.SOURCES_NET[rec["src"]]
print(f"phasor stack {zt.shape}")

In [ ]:
show_m = [0, cfg.M_FREQ // 2, cfg.M_FREQ - 1]
fig, ax = plt.subplots(len(show_m), 4, figsize=(11.6, 2.65 * len(show_m)))
yx = np.arange(cfg.N_NET)

for r_, m in enumerate(show_m):
    at = np.abs(zt[m]).sum(0) ** 0.5      # sqrt of summed |u|^2 over components
    ap = np.abs(zp[m]).sum(0) ** 0.5
    err = np.abs(zp[m] - zt[m]).sum(0) ** 0.5
    vmax = float(at.max())
    for c_, (img, ttl) in enumerate([(at, "true |u_s|"), (ap, "predicted"),
                                     (err, "|error|")]):
        im = ax[r_, c_].imshow(img, origin="lower", cmap="magma", vmin=0, vmax=vmax)
        ax[r_, c_].contour(yx, yx, chi1, levels=[0.5], colors="c", linewidths=0.9)
        ax[r_, c_].plot(sx_, sy_, "w*", ms=7)
        ax[r_, c_].set(xticks=[], yticks=[])
        ax[r_, c_].grid(False)
        if r_ == 0:
            ax[r_, c_].set_title(ttl, fontsize=9)
    ax[r_, 0].set_ylabel(f"f = {cfg.FREQS[m]:.3f} f_c", fontsize=8.5)
    fig.colorbar(im, ax=ax[r_, 2], fraction=0.046)

    # phase of the x component, only where there is amplitude to have a phase
    ph = np.angle(zp[m, 0] * np.conj(zt[m, 0]))
    mask = np.abs(zt[m, 0]) < 0.05 * np.abs(zt[m, 0]).max()
    ph = np.where(mask, np.nan, ph)
    im2 = ax[r_, 3].imshow(ph, origin="lower", cmap="twilight_shifted",
                           vmin=-np.pi, vmax=np.pi)
    ax[r_, 3].contour(yx, yx, chi1, levels=[0.5], colors="k", linewidths=0.9)
    ax[r_, 3].set(xticks=[], yticks=[])
    ax[r_, 3].grid(False)
    if r_ == 0:
        ax[r_, 3].set_title("phase error, u_x\n(blank where |u| < 5% of max)",
                            fontsize=9)
    fig.colorbar(im2, ax=ax[r_, 3], fraction=0.046)

fig.suptitle(f"worst held-out-source sample {rec['index']}: "
             f"rel-L2 {rec['field']:.3f}", fontsize=10)
fig.tight_layout()
savefig(fig, "04_fig1_wavefield.png")
plt.show()

### Time-domain context

The dataset keeps 64 downsampled velocity snapshots for `N_VIS_SAMPLES = 8` samples per split.
These are the **total** field from the solver, not the network's output -- the network predicts
phasors and has no time axis. They are here because the phasor panels above are hard to read
without knowing what the wave was doing: the frames show the incident front crossing the void,
the scattered wave leaving it, and both being absorbed at the walls.

The snapshots are also why the running DFT exists. With 64 frames over 24 P periods the frame
Nyquist is 1.33 f_c, sitting on top of the operating band, so FFT-ing these frames would alias
exactly where it matters. The phasors are accumulated inside the time loop instead.

In [ ]:
import h5py

with h5py.File(paths["test"], "r") as f:
    vis_idx = f["vis/index"][:] if "vis/index" in f else np.array([], dtype=int)
    if len(vis_idx):
        k = int(np.argmin(np.abs(vis_idx - rec["index"])))
        fr = f["vis/frames"][k]                     # [n_frames, 2, ny, nx]
        vis_sample = int(vis_idx[k])
    else:
        fr = None

if fr is None:
    print("no visualisation frames in this file")
else:
    print(f"frames for sample {vis_sample} "
          f"({'the montage sample' if vis_sample == rec['index'] else 'the nearest one'})"
          f": {fr.shape}, every {cfg.SAVE_EVERY} steps")
    ks = np.linspace(fr.shape[0] * 0.18, fr.shape[0] - 1, 5).astype(int)
    v = np.abs(fr[:, 0]).max()
    fig, ax = plt.subplots(1, len(ks), figsize=(2.15 * len(ks), 2.4))
    for a_, kk in zip(ax, ks):
        a_.imshow(fr[kk, 0], origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
        a_.set(xticks=[], yticks=[],
               title=f"t = {kk * cfg.SAVE_EVERY * cfg.DT:.1f} T_p")
        a_.grid(False)
    fig.suptitle("total v_x, solver frames (not a network output)", fontsize=9)
    fig.tight_layout()
    savefig(fig, "04_frames.png")
    plt.show()

## Figure 2 -- all 32 receivers

The ring is 32 pixels out of 16384, and it is the entire input to the inversion. §11.3 asks
for predicted-vs-true A-scans at all 32 receivers; what the network produces is a phasor, so
this comes in two panels and the distinction between them is worth being explicit about.

**Left: the frequency-domain gather.** Real and imaginary parts against receiver index, at
three frequencies, in physical units (the stored values are divided by the incident domain
maximum, so they are multiplied back). This is the actual predicted quantity, compared without
any further processing. The phase of these numbers is what the inversion minimises.

**Right: a band-limited time reconstruction.** `u(t) = 2 df Re sum_m u_hat_m exp(i omega_m t)`
over the 20 lines, applied *identically* to prediction and truth. It is not the solver's
A-scan: 20 lines spanning 0.66-1.34 f_c give a time resolution of about 1.5 periods, and the
reconstruction has no information outside the band, so the pulse shape is the band's, not the
source's. No attempt is made to recover the absolute time origin. It is included because
agreement here is easier to *see* than agreement in a scatter of complex numbers -- and
because a phase error that the gather hides as a small rotation shows up in the time trace as
a shifted arrival, which is the failure the inversion cares about.

In [ ]:
scale = _np(b["scale"][0])                      # [M], the per-line divisor
ry, rx = _np(recv[:, 0]), _np(recv[:, 1])
gt = zt[:, :, ry, rx] * scale[:, None, None]    # [M, 2, R] physical
gp = zp[:, :, ry, rx] * scale[:, None, None]

fig = plt.figure(figsize=(11.6, 6.2))
gs = fig.add_gridspec(3, 2, width_ratios=[1.0, 1.15])

for r_, m in enumerate(show_m):
    a_ = fig.add_subplot(gs[r_, 0])
    ridx = np.arange(len(ry))
    a_.plot(ridx, gt[m, 0].real, "-", lw=1.2, c="C0", label="true Re u_x")
    a_.plot(ridx, gp[m, 0].real, "--", lw=1.2, c="C1", label="pred Re u_x")
    a_.plot(ridx, gt[m, 0].imag, "-", lw=1.0, c="C2", alpha=0.8, label="true Im u_x")
    a_.plot(ridx, gp[m, 0].imag, "--", lw=1.0, c="C3", alpha=0.8, label="pred Im u_x")
    a_.set_ylabel(f"f = {cfg.FREQS[m]:.3f}", fontsize=8.5)
    if r_ == 0:
        a_.legend(fontsize=6.5, ncol=2)
        a_.set_title("frequency-domain gather, 32 receivers", fontsize=9)
    if r_ == len(show_m) - 1:
        a_.set_xlabel("receiver index (counter-clockwise around the ring)")

# band-limited time reconstruction, same operator on both
om = 2.0 * np.pi * np.asarray(cfg.FREQS)
tt = np.linspace(0.0, 12.0, 900)
kern = np.exp(1j * om[:, None] * tt[None, :])
ut = 2.0 * cfg.DF * np.real(np.tensordot(gt[:, 0, :], kern, axes=(0, 0)))   # [R, T]
up = 2.0 * cfg.DF * np.real(np.tensordot(gp[:, 0, :], kern, axes=(0, 0)))

a_ = fig.add_subplot(gs[:, 1])
step = 1.05 * np.abs(ut).max()
for i in range(ut.shape[0]):
    a_.plot(tt, ut[i] + i * step, "-", lw=0.7, c="C0")
    a_.plot(tt, up[i] + i * step, "--", lw=0.7, c="C1")
a_.plot([], [], "-", c="C0", label="true")
a_.plot([], [], "--", c="C1", label="predicted")
a_.set(xlabel="t (arbitrary origin) / T_p", yticks=[],
       title="band-limited reconstruction, u_x at all 32 receivers\n"
             "(20 lines over 0.66-1.34 f_c: resolution ~1.5 periods)")
a_.legend(fontsize=8, loc="upper right")

fig.tight_layout()
savefig(fig, "04_fig2_receivers.png")
plt.show()

num = np.abs(gp - gt).ravel()
den = np.abs(gt).ravel()
print(f"ring rel-L2 for this sample: "
       f"{np.linalg.norm(num)/np.linalg.norm(den):.4f}")
print(f"ring amplitude is {np.abs(gt).max()/np.abs(zt*scale[:,None,None,None]).max():.4f} "
      f"of the domain maximum -- the two normalisation scales of §5.3 differ by about "
      f"two orders of magnitude, which is why the receiver misfit uses scale_recv "
      f"and the network inputs use scale")

## Figure 3 -- where the error lives in `|k|`

The error field's power, binned by integer radius `|k|` on the `128^2` grid, against the true
field's, both from `rfft2` with `norm='ortho'`. Three vertical lines matter:

- `band_in_modes()`: the mode index the shortest shear wave in the band occupies. Below this
  is physics the network must represent.
- `KMAX = 28`: the truncation. Above it the spectral layers contribute nothing, and whatever
  the network gets right up there comes from the pointwise `1x1` convolutions alone.
- `K_NYQUIST`: the grid's own limit.

The expected picture: relative error roughly flat and small below `band_in_modes`, rising
between there and `KMAX`, and the truncated region carrying little true power -- which is the
justification for truncating at all. What would be a problem is significant *true* power above
`KMAX`, because that is signal the architecture has thrown away, and no amount of training
recovers it. That is also the prediction the `tiny` variant (`KMAX = 16`, inside the band)
is there to test.

In [ ]:
def radial_power(z):
    """[.., 2, ny, nx] complex field -> (k, mean power per integer |k| bin)."""
    t = torch.from_numpy(np.ascontiguousarray(z))
    F_ = torch.fft.rfft2(t.real, norm="ortho") + 1j * torch.fft.rfft2(t.imag,
                                                                     norm="ortho")
    p = _np(F_.abs().pow(2).sum(tuple(range(F_.dim() - 2))))     # [ny, nkx]
    ny, nkx = p.shape
    fy = np.fft.fftfreq(ny) * ny
    fx = np.arange(nkx)
    kk = np.hypot(fy[:, None], fx[None, :])
    kb = np.rint(kk).astype(int)
    nb = kb.max() + 1
    tot = np.bincount(kb.ravel(), weights=p.ravel(), minlength=nb)
    cnt = np.bincount(kb.ravel(), minlength=nb).clip(1)
    return np.arange(nb), tot / cnt


k_t, p_t = radial_power(zt)
k_e, p_e = radial_power(zp - zt)
k_need = band_in_modes()

fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.3))
ax[0].semilogy(k_t, np.maximum(p_t, 1e-24), lw=1.2, label="true field")
ax[0].semilogy(k_e, np.maximum(p_e, 1e-24), lw=1.2, label="error")
ax[1].semilogy(k_t, np.sqrt(np.maximum(p_e, 1e-30) / np.maximum(p_t, 1e-30)),
               lw=1.2, c="C3")
for a_ in ax:
    a_.axvline(k_need, ls=":", c="C2", lw=1.2,
               label=f"band top needs |k| = {k_need:.1f}")
    a_.axvline(cfg.KMAX, ls="--", c="0.3", lw=1.2, label=f"KMAX = {cfg.KMAX}")
    a_.axvline(cfg.K_NYQUIST, ls="-.", c="0.6", lw=1.0,
               label=f"Nyquist = {cfg.K_NYQUIST}")
    a_.set_xlabel("|k| (integer modes on the 128^2 grid)")
ax[0].set(ylabel="mean power per mode", title="spectra, all 20 lines")
ax[0].legend(fontsize=7)
ax[1].set(ylabel="relative error amplitude", title="error / signal vs |k|")
ax[1].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "04_fig3_error_spectrum.png")
plt.show()

above = p_t[cfg.KMAX + 1:].sum() / max(p_t.sum(), 1e-30)
inband = np.sqrt(p_e[:int(k_need) + 1].sum() / max(p_t[:int(k_need) + 1].sum(), 1e-30))
print(f"true power above KMAX (thrown away by truncation): {above:.3e}")
print(f"relative error below |k| = {k_need:.0f} (the physical band): {inband:.4f}")
print(f"relative error over all |k|                       : "
      f"{np.sqrt(p_e.sum()/max(p_t.sum(),1e-30)):.4f}")

## Per-frequency error

`evaluate(..., per_freq=True)` accumulates the numerator and denominator separately per line,
so this is a proper relative error per frequency and not an average of ratios. The top of the
band has the fewest points per wavelength on both grids, so it should be the worst line -- the
same ordering check 5 in notebook 01 found in the solver's own convergence. If the *bottom* of
the band is worst, something is wrong with the deconvolution rather than with the network.

In [ ]:
pf = np.asarray(ev.per_freq)
fig, ax = plt.subplots(figsize=(6.6, 3.1))
ax.bar(cfg.FREQS[:len(pf)], pf, width=0.85 * cfg.DF, color="C0", alpha=0.85)
ax.axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0, label=f"gate {cfg.GATE_REL_L2:.0%}")
ax.axhline(pf.mean(), ls=":", c="0.35", lw=1.0, label=f"mean {pf.mean():.3f}")
ax.set(xlabel="f / f_c", ylabel="rel-L2", title="test error per frequency line")
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "04_per_frequency.png")
plt.show()

table([(f"{m}", f"{cfg.FREQS[m]:.4f}", f"{pf[m]:.4f}",
        "worst" if m == int(pf.argmax()) else "")
       for m in range(len(pf))], ["m", "f / f_c", "rel-L2", ""])

## Throughput

The point of the surrogate is that the inversion can afford tens of thousands of forward
evaluations. This measures what one costs, and compares it against the solver time recorded by
notebook 02 -- so the speedup quoted is measured on this machine rather than asserted.

Stage 1 alone evaluates `SCREEN_GRID^2 = 256` candidates over 6 frequencies, and stages 2 and
3 add a few thousand more. At solver cost that is weeks per inversion.

In [ ]:
with torch.no_grad():
    for _ in range(3):
        model(x1)
    if DEV.startswith("cuda"):
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    n_rep = 20
    for _ in range(n_rep):
        model(x1)
    if DEV.startswith("cuda"):
        torch.cuda.synchronize()
    per_call = (time.perf_counter() - t0) / n_rep

rows_per_call = x1.shape[0]
print(f"{rows_per_call} rows (1 geometry x {cfg.M_FREQ} lines) per call")
print(f"  {per_call*1e3:8.2f} ms per call")
print(f"  {per_call/rows_per_call*1e3:8.3f} ms per (geometry, frequency)")
print(f"  {rows_per_call/per_call:8.0f} rows/s")

solver_s = None
gp_ = E.results / "02_dataset_generation.json"
if gp_.exists():
    thr = json.loads(gp_.read_text()).get("throughput_cell_steps_per_s")
    if thr:
        # one geometry is N_FINE_TOTAL^2 cells x NT steps, and yields all M_FREQ
        # lines at once because the DFT runs inside the time loop
        solver_s = cfg.N_FINE_TOTAL ** 2 * cfg.NT / thr
if solver_s:
    n_cand = cfg.SCREEN_GRID ** 2
    print(f"\nsolver: {solver_s:.2f} s per geometry (all {cfg.M_FREQ} lines), from "
          f"notebook 02's measured throughput")
    print(f"speedup: {solver_s/per_call:,.0f}x per forward evaluation")
    print(f"stage 1's {n_cand} candidates: {n_cand*per_call:.2f} s surrogate "
          f"vs {n_cand*solver_s/3600:.1f} h solver")
else:
    print("\n(no throughput in 02_dataset_generation.json -- run notebook 02 on a "
          "GPU for the solver baseline)")

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "checkpoint": str(CKPT),
    "arch": meta["arch"], "epoch": meta["epoch"],
    "gates": ev.gates(),
    "test": dict(rel_l2=ev.rel_l2, ring=ev.ring_rel_l2, phase=ev.phase_periods,
                 per_freq=ev.per_freq),
    "per_sample": dict(
        n=len(fe), mean=float(fe.mean()), median=float(np.median(fe)),
        p95=float(np.percentile(fe, 95)), max=float(fe.max()),
        trained_mean=float(fe[~held].mean()) if (~held).any() else None,
        heldout_mean=float(fe[held].mean()) if held.any() else None,
        heldout_penalty=float(pen) if held.any() else None),
    "spectrum": dict(true_power_above_kmax=float(above),
                     rel_error_in_band=float(inband),
                     kmax=cfg.KMAX, band_in_modes=float(k_need)),
    "throughput": dict(seconds_per_call=per_call, rows_per_call=int(rows_per_call),
                       solver_seconds_per_sample=solver_s),
    "worst_heldout_sample": rec,
}
dump(record, "04_forward_eval.json")

ts.close()
one.close()
print()
for k, v in ev.gates().items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
if all(ev.gates().values()):
    print("\nThe forward operator is good enough to invert against.  Notebook 05.")
else:
    print("\nSTOP.  §11.3: the forward gates come before the inverse ones.  An\n"
          "inversion against a surrogate that is 10% wrong at the receivers will\n"
          "converge confidently to the wrong defect, and the misfit at the answer\n"
          "will look plausible because the model error is systematic.")